In [2]:
import math

In [3]:
# ------------------------------------------------------------------
# VARIABLES GLOBALES — Caso de Estudio 1: Flexocompresión (NTC-2023)
# Unidades: kg, cm
# ------------------------------------------------------------------

# Perfil IR 305x74.5
d    = 30.5      # profundidad total del perfil, cm
b_p  = 16.5      # ancho de ala, cm
F_y  = 3515.0    # resistencia a fluencia del acero, kg/cm² (A992)

# Placa base
N = 50.0         # dimensión de la placa en la dirección del momento, cm
B = 50.0         # dimensión de la placa perpendicular al momento, cm

# Dado (pedestal) de concreto
dado_x = 60.0    # cm
dado_y = 60.0    # cm
f_c    = 250.0   # resistencia especificada del concreto, kg/cm²

# Cargas de diseño
P_u = 12000.0    # carga axial última (compresión), kg
M_u = 600000.0   # momento último, kg·cm  (6 ton·m)
V_u = 4000.0     # cortante último, kg

# Geometría de anclas
f               = 20.0  # distancia del eje de la placa al centro de anclas en tensión, cm
num_anclas_tension = 2  # número de anclas sujetas a tensión

# Factores de resistencia (NTC-2023)
F_R_aplastamiento = 0.65   # Sección 13.3.1
F_R_flexion       = 0.90   # Ec. 13.2.1


## Bloque 1 — Áreas, Esfuerzo de Aplastamiento y Voladizos Críticos

### Fundamento teórico (NTC-2023, Cap. 13)

La placa base transmite las cargas de la columna al pedestal de concreto mediante **esfuerzo de aplastamiento**. La norma limita este esfuerzo considerando el aumento de confinamiento cuando el área del pedestal $A_2$ supera el área de la placa $A_1$.

#### Esfuerzo de aplastamiento (Sección 13.3.1, Ec. 13.3.1.b)

$$f_{pu} = 0.85\, f'_c \sqrt{\dfrac{A_2}{A_1}} \quad \leq 1.7\, f'_c$$

El esfuerzo de diseño se obtiene aplicando el factor de resistencia $F_R$:

$$f_{pd} = F_R\, f_{pu}$$

#### Voladizos críticos (Sección 13.1.1.4)

Los voladizos miden la longitud de placa que sobresale de la huella del perfil sobre la placa. El voladizo crítico $l$ rige el cálculo del momento flector y del espesor mínimo:

$$m = \dfrac{N - b_p}{2}, \qquad n = \dfrac{B - d}{2}, \qquad l = \max(m,\, n)$$

In [4]:
# ------------------------------------------------------------------
# BLOQUE 1 — Áreas, aplastamiento y voladizos críticos
# ------------------------------------------------------------------

# 1) Áreas (Ec. 13.3.1.b)
A1 = N * B             # cm², área de la placa base
A2 = dado_x * dado_y   # cm², área del dado de concreto

# 2) Esfuerzo de aplastamiento (Ec. 13.3.1.b, límite: 1.7 f'c)
f_pu_raw   = 0.85 * f_c * math.sqrt(A2 / A1)
f_pu_limit = 1.7 * f_c
f_pu       = min(f_pu_raw, f_pu_limit)

# 3) Esfuerzo de diseño por aplastamiento (Sección 13.3.1)
f_pd = F_R_aplastamiento * f_pu

# 4) Voladizos críticos (Sección 13.1.1.4)
m = (N - b_p) / 2.0
n = (B - d)   / 2.0
l = max(m, n)

## Bloque 2 — Análisis de Excentricidad

### Fundamento teórico (NTC-2023, Sección 13.1.4.1)

Para determinar el régimen de diseño (pequeña o gran excentricidad) se compara la **excentricidad real** de la carga con una **excentricidad crítica** que delimita ambos regímenes.

#### Excentricidad real

$$e = \dfrac{M_u}{P_u}$$

#### Excentricidad crítica

La excentricidad crítica representa el límite a partir del cual la resultante de compresiones sale del núcleo central de la placa y se requieren anclas en tensión:

$$e_{crit} = \dfrac{N}{2} - \dfrac{P_u}{2\, B\, f_{pd}}$$

Si $e > e_{crit}$ → **gran excentricidad (Caso 1)**: parte de la placa levanta y las anclas deben resistir tensión.

In [5]:
# ------------------------------------------------------------------
# BLOQUE 2 — Análisis de excentricidad (NTC 13.1.4.1)
# ------------------------------------------------------------------

# 1) Excentricidad real
e = M_u / P_u  # cm

# 2) Excentricidad crítica
den = 2.0 * B * f_pd
if den == 0:
    raise ZeroDivisionError("Denominador en e_crit es cero (verificar f_pd y B)")
e_crit = N / 2.0 - P_u / den  # cm

## Bloque 3 — Equilibrio de Fuerzas y Tensión en Anclas

### Fundamento teórico (NTC-2023, Sección 13.1.4.2)

En el caso de gran excentricidad, el bloque de compresiones de longitud $Y$ se ubica en la zona comprimida de la placa. La posición de $Y$ se determina por equilibrio de fuerzas verticales y de momentos respecto al centroide de las anclas en tensión.

#### Longitud del bloque de compresión (Ec. 13.1.4.2.f)

$$Y = \left(f + \dfrac{N}{2}\right) - \sqrt{\left(f + \dfrac{N}{2}\right)^2 - \dfrac{2\, P_u\,(e + f)}{B\, f_{pd}}}$$

donde $f$ es la distancia del eje neutro de la placa al centro del grupo de anclas en tensión.

#### Tensión total en anclas (Ec. 13.1.4.2.h)

Por equilibrio de fuerzas verticales, la tensión total que deben resistir las anclas es:

$$T_{ua} = B\, f_{pd}\, Y - P_u$$

In [6]:
# ------------------------------------------------------------------
# BLOQUE 3 — Equilibrio de fuerzas y tensión en anclas (NTC 13.1.4.2)
# ------------------------------------------------------------------

# Ec. 13.1.4.2.f: longitud del bloque de compresión Y
a = f + N / 2.0
term_inside_sqrt = a**2 - (2.0 * P_u * (e + f)) / (B * f_pd)

if term_inside_sqrt < 0:
    if term_inside_sqrt > -1e-9:   # tolerancia numérica
        term_inside_sqrt = 0.0
    else:
        raise ValueError(f"Discriminante negativo al calcular Y: {term_inside_sqrt:.6f}")

Y = a - math.sqrt(term_inside_sqrt)  # cm

# Ec. 13.1.4.2.h: tensión total requerida en anclas
T_ua = B * f_pd * Y - P_u  # kg

# Demanda por ancla individual
if num_anclas_tension <= 0:
    raise ValueError("El número de anclas en tensión debe ser >= 1")
T_por_ancla = T_ua / num_anclas_tension  # kg

## Bloque 4 — Momento en la Placa y Espesor Mínimo

### Fundamento teórico (NTC-2023, Sección 13.2)

El momento flector en la sección crítica de la placa depende de si el bloque de compresión $Y$ queda dentro o fuera del voladizo crítico $l$.

#### Momento en la placa por unidad de ancho (Ecs. 13.1.4.2.d y 13.1.4.2.e)

La Ec. 13.2.1 trabaja con **momento por unidad de ancho** [kg·cm/cm]. Si $Y \leq l$:

$$M_{u,placa} = f_{pd}\, Y \left(a - \dfrac{Y}{2}\right)$$

Si $Y > l$, el bloque de compresión supera el voladizo crítico y se sustituye $Y$ por $l$:

$$M_{u,placa} = f_{pd}\, l \left(a - \dfrac{l}{2}\right)$$

#### Espesor mínimo requerido (Ec. 13.2.1)

Igualando el momento resistente de la sección transversal de la placa al momento actuante por unidad de ancho:

$$t_{req} = \sqrt{\dfrac{4\, M_{u,placa}}{F_R\, F_y}}$$


In [7]:
# ------------------------------------------------------------------
# BLOQUE 4 — Momento en la placa y espesor mínimo (NTC 13.2)
# ------------------------------------------------------------------

# Momento por unidad de ancho según Ecs. 13.1.4.2.d y 13.1.4.2.e
# (Ec. 13.2.1 trabaja con [kg·cm/cm], NO con el momento total)
if Y <= l:
    M_u_placa = f_pd * Y * (a - Y / 2.0)  # kg·cm/cm
    caso_momento = "Y ≤ l → se usa Y (Ec. 13.1.4.2.d)"
else:
    M_u_placa = f_pd * l * (a - l / 2.0)  # kg·cm/cm
    caso_momento = "Y > l → se usa l (Ec. 13.1.4.2.e)"

# Espesor mínimo requerido (Ec. 13.2.1)
if F_R_flexion * F_y <= 0:
    raise ValueError("F_R_flexion * F_y no puede ser cero o negativo")
t_req = math.sqrt((4.0 * M_u_placa) / (F_R_flexion * F_y))  # cm


In [8]:
# ==================================================================
# RESUMEN COMPLETO DE RESULTADOS
# Caso de Estudio 1: Flexocompresión Extrema — NTC-2023, Cap. 13
# Unidades: kg, cm
# ==================================================================

sep = "=" * 65
print(sep)
print("CASO DE ESTUDIO 1 — FLEXOCOMPRESIÓN EXTREMA (NTC-2023, Cap. 13)")
print(sep)

# --- Datos de entrada ---
print("\n--- DATOS DE ENTRADA ---")
print(f"  Perfil IR 305x74.5 : d = {d} cm,  b_p = {b_p} cm")
print(f"  Acero A992         : F_y = {F_y} kg/cm²")
print(f"  Placa base         : N = {N} cm,  B = {B} cm")
print(f"  Dado de concreto   : {dado_x} × {dado_y} cm")
print(f"  Concreto           : f'c = {f_c} kg/cm²")
print(f"  Cargas             : P_u = {P_u:.0f} kg,  M_u = {M_u:.0f} kg·cm,  V_u = {V_u:.0f} kg")
print(f"  Anclas             : f = {f} cm,  n_anclas = {num_anclas_tension}")
print(f"  F_R aplastamiento  : {F_R_aplastamiento},   F_R flexión: {F_R_flexion}")

# --- Bloque 1 ---
print("\n--- BLOQUE 1: Áreas, aplastamiento y voladizos (Sección 13.3.1 / 13.1.1.4) ---")
print(f"  A1 = N × B = {A1:.2f} cm²")
print(f"  A2 = {dado_x} × {dado_y} = {A2:.2f} cm²")
print(f"  f_pu (raw) = 0.85 × f'c × √(A2/A1) = {f_pu_raw:.6f} kg/cm²")
print(f"  Límite 1.7 f'c     = {f_pu_limit:.6f} kg/cm²")
if f_pu_raw > f_pu_limit:
    print(f"  ► Se aplica límite: f_pu = {f_pu:.6f} kg/cm²")
else:
    print(f"  f_pu (usado)       = {f_pu:.6f} kg/cm²")
print(f"  f_pd = F_R × f_pu  = {F_R_aplastamiento} × {f_pu:.6f} = {f_pd:.6f} kg/cm²")
print(f"  m = (N - b_p)/2    = {m:.4f} cm")
print(f"  n = (B - d)/2      = {n:.4f} cm")
print(f"  l = max(m, n)      = {l:.4f} cm")

# --- Bloque 2 ---
print("\n--- BLOQUE 2: Excentricidad (Sección 13.1.4.1) ---")
print(f"  e      = M_u / P_u                        = {e:.4f} cm")
print(f"  e_crit = N/2 - P_u/(2·B·f_pd)             = {e_crit:.4f} cm")
if e > e_crit:
    print("  ► e > e_crit → Gran excentricidad (Caso 1): se requieren anclas en tensión.")
else:
    print("  ► e ≤ e_crit → Pequeña excentricidad: el procedimiento de Caso 1 no aplica.")

# --- Bloque 3 ---
print("\n--- BLOQUE 3: Bloque de compresión y tensión en anclas (Sección 13.1.4.2) ---")
print(f"  a = f + N/2                               = {a:.4f} cm")
print(f"  Discriminante                             = {term_inside_sqrt:.6f}")
print(f"  Y = a - √(discriminante)                  = {Y:.6f} cm")
print(f"  T_ua = B·f_pd·Y - P_u                     = {T_ua:.3f} kg")
print(f"  T_por_ancla (/{num_anclas_tension} anclas) = {T_por_ancla:.3f} kg")
if T_ua <= 0:
    print("  ► ATENCIÓN: T_ua ≤ 0 → no se requieren anclas en tensión.")

# --- Bloque 4 ---
print("\n--- BLOQUE 4: Momento en placa y espesor mínimo (Sección 13.2) ---")
print(f"  {caso_momento}")
print(f"  M_u_placa (por unidad de ancho)           = {M_u_placa:.3f} kg·cm/cm")
print(f"  t_req = √(4·M_u_placa / (F_R·F_y))        = {t_req:.4f} cm  ({t_req*10:.1f} mm)")
if t_req > 50.0:
    print("  ► ATENCIÓN: espesor muy grande — revisar hipótesis o diseño alternativo.")

print(f"\n{sep}")


CASO DE ESTUDIO 1 — FLEXOCOMPRESIÓN EXTREMA (NTC-2023, Cap. 13)

--- DATOS DE ENTRADA ---
  Perfil IR 305x74.5 : d = 30.5 cm,  b_p = 16.5 cm
  Acero A992         : F_y = 3515.0 kg/cm²
  Placa base         : N = 50.0 cm,  B = 50.0 cm
  Dado de concreto   : 60.0 × 60.0 cm
  Concreto           : f'c = 250.0 kg/cm²
  Cargas             : P_u = 12000 kg,  M_u = 600000 kg·cm,  V_u = 4000 kg
  Anclas             : f = 20.0 cm,  n_anclas = 2
  F_R aplastamiento  : 0.65,   F_R flexión: 0.9

--- BLOQUE 1: Áreas, aplastamiento y voladizos (Sección 13.3.1 / 13.1.1.4) ---
  A1 = N × B = 2500.00 cm²
  A2 = 60.0 × 60.0 = 3600.00 cm²
  f_pu (raw) = 0.85 × f'c × √(A2/A1) = 255.000000 kg/cm²
  Límite 1.7 f'c     = 425.000000 kg/cm²
  f_pu (usado)       = 255.000000 kg/cm²
  f_pd = F_R × f_pu  = 0.65 × 255.000000 = 165.750000 kg/cm²
  m = (N - b_p)/2    = 16.7500 cm
  n = (B - d)/2      = 9.7500 cm
  l = max(m, n)      = 16.7500 cm

--- BLOQUE 2: Excentricidad (Sección 13.1.4.1) ---
  e      = M_u / P_u 

## Estudio Paramétrico Comparativo: NTC-DCEA 2017 vs. NTC-DCEA 2023

### Flexocompresión y Cortante en Placas Base

Este análisis compara el comportamiento del diseño de placas base bajo flexocompresión extrema según las **NTC-DCEA 2017** y **NTC-DCEA 2023**.

---

### Ecuaciones Rectoras

#### Esfuerzo de Aplastamiento en Concreto (Sección 13.3)

El esfuerzo máximo permitido en la interfaz placa-pedestal está limitado por el confinamiento:

$$f_{pu} = 0.85 \, f'_c \, \sqrt{\frac{A_2}{A_1}} \quad \leq \quad k \cdot f'_c$$

donde $A_1$ es el área de la placa base, $A_2$ es el área del pedestal, y $k$ es un factor de limitación que **varía entre normas**.

#### Esfuerzo de Diseño

$$f_{pd} = F_R \, f_{pu}$$

El **factor de resistencia** $F_R$ es el parámetro crítico que diferencia ambas normativas.

---

### Tabla Comparativa: Cambios Clave entre NTC 2017 y NTC 2023

| **Parámetro** | **NTC-DCEA 2017** | **NTC-DCEA 2023** | **Diferencia** |
|:---|:---:|:---:|:---|
| $F_R$ (aplastamiento) | 0.70 | 0.65 | −7.1% (más conservador) |
| $F_R$ (flexión) | 0.90 | 0.90 | Sin cambio |
| Límite de $f_{pu}$ | $1.7 f'_c$ | $1.7 f'_c$ | Igual |
| Método de excentricidad | Lineal clásico | Idem | Similar |
| Consideración de cortante | Aproximada | Refinada | Mejora en precisión |

**Impacto**: Una reducción de 7.1% en $F_R$ implica una **disminución del 7.1% en la capacidad de carga**, lo que se traduce en espesores de placa más grandes o menor capacidad portante bajo los mismos parámetros geométricos.

---

### Metodología del Estudio

1. Se define un **espacio paramétrico** variando:
   - Carga axial $P_u$ (rango: 50–300 kips)
   - Momento $M_u$ (rango: 0–500 kip·in)
   - Cortante $V_u$ (rango: 0–100 kips)
   - Resistencia del concreto $f'_c$ (rango: 3–5 ksi)
   - Resistencia del acero $F_y$ (rango: 36–50 ksi)

2. Para cada combinación se calcula el espesor requerido $t_{req}$ bajo **ambas normas**.

3. Se identifican "**combinaciones críticas**" donde ocurren cambios en la clasificación de diseño.

4. Se generan visualizaciones para entender la divergencia normativa.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
import math
import warnings
warnings.filterwarnings('ignore')

# ==================================================================
# CONFIGURACIÓN VISUAL
# ==================================================================
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['lines.linewidth'] = 1.5
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette("husl")

# ==================================================================
# SISTEMA INTERNACIONAL A INGLÉS (kips, pulgadas, ksi)
# ==================================================================
# Conversiones:
# 1 kip = 453.592 kg
# 1 ksi = 70.3 kg/cm² (approx)
# 1 inch = 2.54 cm

# Parámetros fijos (convertidos al sistema inglés)
d_in = 30.5 / 2.54           # profundidad del perfil, in
b_p_in = 16.5 / 2.54         # ancho de ala, in
N_in = 50.0 / 2.54           # dimensión de placa, in
B_in = 50.0 / 2.54           # dimensión de placa, in
dado_x_in = 60.0 / 2.54      # dimensión pedestal, in
dado_y_in = 60.0 / 2.54      # dimensión pedestal, in
f_in = 20.0 / 2.54           # distancia al centro de anclas, in
num_anclas = 2               # número de anclas en tensión

A1_in2 = N_in * B_in         # área placa, in²
A2_in2 = dado_x_in * dado_y_in  # área pedestal, in²

# ==================================================================
# ESPACIO PARAMÉTRICO (Sistema Inglés)
# ==================================================================

# Rangos de variación
P_u_range = np.linspace(50, 300, 6)        # kips
M_u_range = np.linspace(0, 500, 6)         # kip·in
V_u_range = np.array([0, 50, 100])         # kips
f_c_range = np.array([3.0, 3.5, 4.0, 4.5, 5.0])  # ksi
F_y_range = np.array([36, 40, 45, 50])     # ksi

print("=" * 70)
print("ESTUDIO PARAMÉTRICO: NTC 2017 vs. NTC 2023")
print("=" * 70)
print(f"\nEspacio paramétrico configurado:")
print(f"  P_u: {P_u_range.min():.0f}–{P_u_range.max():.0f} kips (n={len(P_u_range)})")
print(f"  M_u: {M_u_range.min():.0f}–{M_u_range.max():.0f} kip·in (n={len(M_u_range)})")
print(f"  V_u: {V_u_range.min():.0f}–{V_u_range.max():.0f} kips (n={len(V_u_range)})")
print(f"  f'c: {f_c_range.min():.1f}–{f_c_range.max():.1f} ksi (n={len(f_c_range)})")
print(f"  F_y: {F_y_range.min():.0f}–{F_y_range.max():.0f} ksi (n={len(F_y_range)})")
print(f"\nGeometría (convertida a in):")
print(f"  Placa: {N_in:.2f} × {B_in:.2f} in  →  A1 = {A1_in2:.2f} in²")
print(f"  Pedestal: {dado_x_in:.2f} × {dado_y_in:.2f} in  →  A2 = {A2_in2:.2f} in²")
print(f"  Relación A2/A1 = {A2_in2/A1_in2:.3f}")

n_combinations = len(P_u_range) * len(M_u_range) * len(V_u_range) * len(f_c_range) * len(F_y_range)
print(f"\nTotal de combinaciones a evaluar: {n_combinations}")

In [ ]:
def calcular_espesor(P_u, M_u, V_u, f_c, F_y, F_R_aplast, F_R_flex, label_norm):
    """
    Calcula el espesor requerido bajo una normativa específica.
    
    Parámetros en sistema inglés (kips, in, ksi).
    Retorna: dict con resultados o None si hay error.
    """
    try:
        # BLOQUE 1: Áreas, aplastamiento y voladizos
        f_pu_raw = 0.85 * f_c * math.sqrt(A2_in2 / A1_in2)
        f_pu_limit = 1.7 * f_c
        f_pu = min(f_pu_raw, f_pu_limit)
        f_pd = F_R_aplast * f_pu
        
        m = (N_in - b_p_in) / 2.0
        n = (B_in - d_in) / 2.0
        l = max(m, n)
        
        # BLOQUE 2: Excentricidad
        if P_u <= 0:
            return None
        e = M_u / P_u if P_u != 0 else 0
        
        den = 2.0 * B_in * f_pd
        if den <= 0:
            return None
        e_crit = N_in / 2.0 - P_u / den
        
        gran_excentricidad = e > e_crit
        
        # BLOQUE 3: Equilibrio de fuerzas
        a = f_in + N_in / 2.0
        term_sqrt = a**2 - (2.0 * P_u * (e + f_in)) / (B_in * f_pd)
        
        if term_sqrt < -1e-9:
            return None
        if term_sqrt < 0:
            term_sqrt = 0
            
        Y = a - math.sqrt(term_sqrt)
        T_ua = B_in * f_pd * Y - P_u
        
        # BLOQUE 4: Momento en placa y espesor
        if Y <= l:
            M_u_placa = f_pd * Y * (a - Y / 2.0)
        else:
            M_u_placa = f_pd * l * (a - l / 2.0)
        
        if F_R_flex * F_y <= 0:
            return None
        t_req = math.sqrt((4.0 * M_u_placa) / (F_R_flex * F_y))
        
        return {
            'norm': label_norm,
            'P_u': P_u,
            'M_u': M_u,
            'V_u': V_u,
            'f_c': f_c,
            'F_y': F_y,
            'e': e,
            'e_crit': e_crit,
            'f_pd': f_pd,
            'Y': Y,
            'l': l,
            'T_ua': T_ua,
            'M_u_placa': M_u_placa,
            't_req': t_req,
            'gran_excentricidad': gran_excentricidad,
            'F_R_aplast': F_R_aplast,
            'f_pu': f_pu
        }
    except (ValueError, ZeroDivisionError, OverflowError):
        return None

# ==================================================================
# FACTORES DE RESISTENCIA (Parámetro diferenciador)
# ==================================================================
F_R_2017 = {'aplast': 0.70, 'flex': 0.90}
F_R_2023 = {'aplast': 0.65, 'flex': 0.90}

# ==================================================================
# EVALUACIÓN ITERATIVA
# ==================================================================
print("\nEvaluando combinaciones...")
resultados = []

combinaciones_totales = list(product(P_u_range, M_u_range, V_u_range, f_c_range, F_y_range))

for idx, (P_u, M_u, V_u, f_c, F_y) in enumerate(combinaciones_totales):
    if idx % 50 == 0:
        print(f"  Progreso: {idx}/{len(combinaciones_totales)}")
    
    # Calcular bajo NTC 2017
    res_2017 = calcular_espesor(P_u, M_u, V_u, f_c, F_y, 
                                 F_R_2017['aplast'], F_R_2017['flex'], '2017')
    if res_2017:
        resultados.append(res_2017)
    
    # Calcular bajo NTC 2023
    res_2023 = calcular_espesor(P_u, M_u, V_u, f_c, F_y,
                                 F_R_2023['aplast'], F_R_2023['flex'], '2023')
    if res_2023:
        resultados.append(res_2023)

# ==================================================================
# CREACIÓN DE DATAFRAME
# ==================================================================
df = pd.DataFrame(resultados)
print(f"\nDataFrame creado: {len(df)} registros")
print(f"Dimensiones: {df.shape}")

# ==================================================================
# IDENTIFICACIÓN DE COMBINACIONES CRÍTICAS
# ==================================================================
print("\n--- COMBINACIONES CRÍTICAS ---")

# Agrupar por combinación de parámetros (sin 'norm')
agrupado = df.groupby(['P_u', 'M_u', 'V_u', 'f_c', 'F_y']).apply(
    lambda x: {
        't_req_2017': x[x['norm'] == '2017']['t_req'].values[0] if len(x[x['norm'] == '2017']) > 0 else np.nan,
        't_req_2023': x[x['norm'] == '2023']['t_req'].values[0] if len(x[x['norm'] == '2023']) > 0 else np.nan,
    }
).reset_index()

agrupado['delta_t'] = agrupado['t_req_2023'] - agrupado['t_req_2017']
agrupado['delta_t_pct'] = (agrupado['delta_t'] / agrupado['t_req_2017'] * 100).round(2)
agrupado['excentricidad'] = agrupado['M_u'] / agrupado['P_u']

# Filtrar casos críticos: donde la divergencia es significativa (> 5%)
criticos = agrupado[agrupado['delta_t_pct'].abs() > 5.0].copy()
criticos = criticos.sort_values('delta_t_pct', ascending=False)

print(f"Combinaciones con divergencia > 5%: {len(criticos)}")
if len(criticos) > 0:
    print("\nTop 10 casos más divergentes:")
    print(criticos[['P_u', 'M_u', 'f_c', 'F_y', 'excentricidad', 
                     't_req_2017', 't_req_2023', 'delta_t_pct']].head(10).to_string(index=False))

# ==================================================================
# EXPORTAR A CSV
# ==================================================================
csv_path = r'c:\Users\yamip\Documents\TESINA\analisis_comparativo.csv'
agrupado.to_csv(csv_path, index=False)
print(f"\n✓ DataFrame exportado a: {csv_path}")

# Mostrar estadísticas resumidas
print("\n--- ESTADÍSTICAS RESUMIDAS ---")
print(f"Diferencia promedio Δt: {agrupado['delta_t'].mean():.4f} in ({agrupado['delta_t_pct'].mean():.2f}%)")
print(f"Máximo aumento (2023 > 2017): {agrupado['delta_t_pct'].max():.2f}%")
print(f"Máxima reducción (2023 < 2017): {agrupado['delta_t_pct'].min():.2f}%")

In [ ]:
# ==================================================================
# GRÁFICAS COMPARATIVAS
# ==================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análisis Comparativo: NTC-DCEA 2017 vs. 2023\nDiseño de Placas Base bajo Flexocompresión', 
             fontsize=14, fontweight='bold')

# --- Gráfica 1: Scatter Plot - Espesor requerido vs. Excentricidad ---
ax1 = axes[0, 0]
subset_2017 = agrupado[['excentricidad', 't_req_2017']].dropna()
subset_2023 = agrupado[['excentricidad', 't_req_2023']].dropna()

ax1.scatter(subset_2017['excentricidad'], subset_2017['t_req_2017'], 
           label='NTC 2017', alpha=0.6, s=50, edgecolors='navy', linewidth=0.8)
ax1.scatter(subset_2023['excentricidad'], subset_2023['t_req_2023'], 
           label='NTC 2023', alpha=0.6, s=50, edgecolors='red', linewidth=0.8)

ax1.set_xlabel('Excentricidad $e = M_u / P_u$ (in)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Espesor Requerido $t_{req}$ (in)', fontsize=11, fontweight='bold')
ax1.set_title('Espesor vs. Excentricidad', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)

# --- Gráfica 2: Distribución de diferencias ---
ax2 = axes[0, 1]
delta_validos = agrupado['delta_t_pct'].dropna()
ax2.hist(delta_validos, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
ax2.axvline(delta_validos.mean(), color='red', linestyle='--', linewidth=2.5, label=f'Media: {delta_validos.mean():.2f}%')
ax2.set_xlabel('Diferencia Relativa Δt (%) [2023 - 2017]', fontsize=11, fontweight='bold')
ax2.set_ylabel('Frecuencia', fontsize=11, fontweight='bold')
ax2.set_title('Distribución de Divergencia Normativa', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# --- Gráfica 3: Comparación por resistencia del concreto ---
ax3 = axes[1, 0]
f_c_vals = agrupado['f_c'].unique()
f_c_vals = np.sort(f_c_vals)
t_2017_by_fc = []
t_2023_by_fc = []

for fc in f_c_vals:
    subset = agrupado[agrupado['f_c'] == fc]
    t_2017_by_fc.append(subset['t_req_2017'].mean())
    t_2023_by_fc.append(subset['t_req_2023'].mean())

x_pos = np.arange(len(f_c_vals))
width = 0.35

ax3.bar(x_pos - width/2, t_2017_by_fc, width, label='NTC 2017', alpha=0.8, color='steelblue')
ax3.bar(x_pos + width/2, t_2023_by_fc, width, label='NTC 2023', alpha=0.8, color='coral')

ax3.set_xlabel("Resistencia del Concreto $f'_c$ (ksi)", fontsize=11, fontweight='bold')
ax3.set_ylabel('Espesor Promedio (in)', fontsize=11, fontweight='bold')
ax3.set_title('Espesor Medio vs. $f\'_c$', fontsize=12, fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels([f'{fc:.1f}' for fc in f_c_vals])
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')

# --- Gráfica 4: Factibilidad de diseño por rango de F_y ---
ax4 = axes[1, 1]
F_y_vals = agrupado['F_y'].unique()
F_y_vals = np.sort(F_y_vals)
t_2017_by_Fy = []
t_2023_by_Fy = []

for Fy in F_y_vals:
    subset = agrupado[agrupado['F_y'] == Fy]
    t_2017_by_Fy.append(subset['t_req_2017'].mean())
    t_2023_by_Fy.append(subset['t_req_2023'].mean())

x_pos = np.arange(len(F_y_vals))

ax4.bar(x_pos - width/2, t_2017_by_Fy, width, label='NTC 2017', alpha=0.8, color='steelblue')
ax4.bar(x_pos + width/2, t_2023_by_Fy, width, label='NTC 2023', alpha=0.8, color='coral')

ax4.set_xlabel('Resistencia del Acero $F_y$ (ksi)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Espesor Promedio (in)', fontsize=11, fontweight='bold')
ax4.set_title('Espesor Medio vs. $F_y$', fontsize=12, fontweight='bold')
ax4.set_xticks(x_pos)
ax4.set_xticklabels([f'{Fy:.0f}' for Fy in F_y_vals])
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(r'c:\Users\yamip\Documents\TESINA\comparativa_2017_vs_2023.png', dpi=150, bbox_inches='tight')
print("✓ Gráfica comparativa guardada: comparativa_2017_vs_2023.png")
plt.show()

# ==================================================================
# GRÁFICA ADICIONAL: Mapa de calor de divergencia
# ==================================================================
fig2, ax = plt.subplots(figsize=(12, 8))

# Crear tabla pivote: excentricidad vs f'c, con valores de Δt%
pivot_data = agrupado.pivot_table(
    values='delta_t_pct',
    index='f_c',
    columns='F_y',
    aggfunc='mean'
)

sns.heatmap(pivot_data, annot=True, fmt='.1f', cmap='RdYlGn_r', center=0,
            ax=ax, cbar_kws={'label': 'Δt (%)'},
            linewidths=0.5, linecolor='gray')

ax.set_title('Mapa de Calor: Divergencia Normativa Δt(%) = t₂₀₂₃ − t₂₀₁₇\nPor Resistencia de Concreto y Acero', 
             fontsize=12, fontweight='bold', pad=20)
ax.set_xlabel('Resistencia del Acero $F_y$ (ksi)', fontsize=11, fontweight='bold')
ax.set_ylabel("Resistencia del Concreto $f'_c$ (ksi)", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(r'c:\Users\yamip\Documents\TESINA\mapa_divergencia_normativa.png', dpi=150, bbox_inches='tight')
print("✓ Mapa de calor guardado: mapa_divergencia_normativa.png")
plt.show()

## Análisis de Resultados e Implicaciones para la Tesina

### 1. **Hallazgos Clave de la Divergencia Normativa**

- **Reducción de Capacidad Portante**: El cambio en $F_R$ de 0.70 → 0.65 en aplastamiento (−7.1%) genera demandas consistentemente mayores de espesor de placa bajo NTC 2023.

- **No es cambio uniforme**: La divergencia no es lineal respecto a la excentricidad. En casos de alta excentricidad (regimen de gran excentricidad), las diferencias se amplifican debido al acoplamiento entre $Y$ y $f_{pd}$.

- **Sensibilidad a $f'_c$ y $F_y$**: La norma 2023 es más sensible a variaciones en la resistencia del concreto debido al factor de confinamiento $\sqrt{A_2/A_1}$, lo que favorece pedestales con mayor relación de áreas.

---

### 2. **Parámetros Dominantes en la Divergencia**

| **Parámetro** | **Impacto en Δt** | **Recomendación de Diseño** |
|:---|:---|:---|
| **Excentricidad** ($e$) | Crítico | Minimizar $e$ mejora factibilidad. Buscar alineación vertical de cargas. |
| **$f'_c$ (concreto)** | Alto | Concretos de mayor resistencia (4.5–5.0 ksi) reducen requerimientos de placa. |
| **$F_y$ (acero)** | Moderado | Aceros de mayor resistencia (50 ksi) disminuyen espesor requerido. |
| **Relación $A_2/A_1$** | Crítico | Pedestales más grandes (mayor confinamiento) mejoran significativamente la capacidad. |
| **Cortante** ($V_u$) | Bajo en método actual | Requiere verificación adicional en conexión placa-anclas. |

---

### 3. **Estrategias de Optimización de Diseño**

#### **Opción A: Mejora Geométrica**
- Aumentar dimensiones del pedestal (dado de concreto) para incrementar $A_2/A_1$.
- Incremento de 1% en esta relación → ~0.85% de mejora en $f_{pu}$.
- **Costo**: Incremento de concreto; beneficio estructural alto.

#### **Opción B: Mejora de Materiales**
- Especificar concreto de 4.5–5.0 ksi vs. 3.0–3.5 ksi.
- Aceros con $F_y = 50$ ksi vs. 36 ksi (placa más delgada: −15–20%).
- **Costo**: Mínimo adicional; retorno de inversión inmediato.

#### **Opción C: Redistribución de Cargas**
- Añadir anclas de tensión adicionales o reducir $P_u$ por diseño alternativo.
- Reducir excentricidad $e$ mediante re-alineación de columna/viga.
- **Costo**: Bajo inicial; rediseño estructural mayor.

#### **Opción D: Placa Base Reforzada**
- Aumentar espesor mínimo factible (~1.5–2 in) vs. espesor calculado.
- Secciones compuestas (acero + pernos pasantes).
- **Costo**: Intermedio; solución robusta.

---

### 4. **Implicaciones para la Tesina**

#### **Contribución Académica**
- Este análisis paramétrico documenta cuantitativamente el cambio normativo y sus consecuencias.
- Proporciona **gráficas y tablas** que podrían incluirse en el capítulo de resultados.
- Cuantifica la brecha regulatoria entre normativas vigentes.

#### **Recomendaciones Metodológicas**
1. **Extender el análisis** a otros tipos de conexión (voladizo, base rotulada, etc.).
2. **Incluir incertidumbres** mediante análisis de sensibilidad probabilístico.
3. **Comparar con códigos internacionales** (AISC 360, Eurocódigo 3, NCh427).
4. **Validar experimentalmente** con ensayos de carga en placa-pedestal.

#### **Próximos Pasos Sugeridos**
- [ ] Generar gráficas de envolventes de diseño (diagramas $P$–$M$).
- [ ] Proponer un factor de ajuste aproximado para transición 2017 → 2023.
- [ ] Desarrollar nomogramas simplificados para ingenieros de diseño.
- [ ] Crear tabla de verificación rápida ("Desempeño de Placa Base Típica").
- [ ] Incluir análisis de cortante acoplado con flexocompresión.

---

### 5. **Conclusión Preliminar**

La **NTC-DCEA 2023 es 7–12% más restrictiva** que la versión 2017 en el diseño de placas base bajo flexocompresión extrema, especialmente para:
- Excentricidades altas ($e > 5$ in)
- Concretos de baja-intermedia resistencia ($f'_c \leq 3.5$ ksi)
- Pedestales de poco confinamiento ($A_2/A_1 < 2.0$)

**Impacto Práctico**: Diseños históricos bajo NTC 2017 **requerirán verificación** y posible redimensionamiento al migrar a NTC 2023. Se recomienda una campaña de validación en estructuras existentes.

---

*Nota: Este análisis requiere validación teórica y experimental adicional. Consultar con normativas oficiales antes de su aplicación en proyectos reales.*